# Validações dependentes de outros campos

Até aqui vimos validadores que agem sobre um valor desserializado sem saber nada dos outros campos do modelo.

Mas, ao desserializar, campos definidos **antes** do campo atual (na ordem da classe) já foram desserializados.

O Pydantic permite que um validador customizado receba informação sobre outros campos.

O porém: ele só conhece campos já validados, e somente os que **passaram** na validação.

Funciona assim: o validador pode receber um argumento extra, do tipo `ValidationInfo`.

Vejamos um exemplo:

In [1]:
from pydantic import BaseModel, field_validator, ValidationError, ValidationInfo

In [2]:
class Model(BaseModel):
    field_1: int
    field_2: list[int]
    field_3: str
    field_4: list[str]

    @field_validator("field_3")
    @classmethod
    def validator(cls, value: str, validated_values: ValidationInfo):
        print(f"{value=}")
        print(f"{validated_values=}")
        return value

In [3]:
print(f'Model(field_1=100, field_2=[1, 2, 3], field_3="python", field_4=["a", "b"]): {Model(field_1=100, field_2=[1, 2, 3], field_3="python", field_4=["a", "b"])!r}')

value='python'
validated_values=ValidationInfo(config={'title': 'Model'}, context=None, data={'field_1': 100, 'field_2': [1, 2, 3]}, field_name='field_3')
Model(field_1=100, field_2=[1, 2, 3], field_3="python", field_4=["a", "b"]): Model(field_1=100, field_2=[1, 2, 3], field_3='python', field_4=['a', 'b'])


O objeto `ValidationInfo` tem um atributo `data` — um dicionário com os campos validados definidos **antes** de `field_3` neste modelo.

Lembre que o Pydantic continua validando mesmo quando um campo falha. Então o `ValidationInfo` pode não conter todos os campos
anteriores — só os que de fato passaram na validação.

In [4]:
try:
    Model(field_1=100, field_2=["a", "b"], field_3="python", field_4=["a", "b"])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

value='python'
validated_values=ValidationInfo(config={'title': 'Model'}, context=None, data={'field_1': 100}, field_name='field_3')
ValidationError: 2 validation errors for Model
field_2.0
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='a', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing
field_2.1
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='b', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


`field_2` não aparece no `ValidationInfo`, porque falhou na validação.

Leve essa possibilidade em conta.

Vejamos uma aplicação típica.

Um modelo com data/hora de início e fim: a validação precisa garantir que o fim não seja anterior ao início.

Trazemos de volta os validadores de datetime:

In [5]:
from datetime import datetime
from typing import Annotated, Any

import pytz
from dateutil.parser import parse
from pydantic import AfterValidator, BeforeValidator

def parse_datetime(value: Any):
    if isinstance(value, str):
        try:
            return parse(value)
        except Exception as ex:
            raise ValueError(str(ex))
    return value


def make_utc(dt: datetime) -> datetime:
    if dt.tzinfo is None:
        dt = pytz.utc.localize(dt)
    else:
        dt = dt.astimezone(pytz.utc)
    return dt

DateTimeUTC = Annotated[datetime, BeforeValidator(parse_datetime), AfterValidator(make_utc)]

In [6]:
class Model(BaseModel):
    start_dt: DateTimeUTC
    end_dt: DateTimeUTC

    @field_validator("end_dt")
    @classmethod
    def validate_end_after_start_dt(cls, value: datetime, values: ValidationInfo):
        data = values.data
        if "start_dt" in data:
            if value <= data["start_dt"]:
                raise ValueError("end_dt must come after start_dt")
        # se start_dt falhou na validação, não há muito o que checar aqui:
        # devolve o valor como está
        return value

In [7]:
print(f'Model(start_dt="2020/1/1", end_dt="2020/12/31"): {Model(start_dt="2020/1/1", end_dt="2020/12/31")!r}')

Model(start_dt="2020/1/1", end_dt="2020/12/31"): Model(start_dt=datetime.datetime(2020, 1, 1, 0, 0, tzinfo=<UTC>), end_dt=datetime.datetime(2020, 12, 31, 0, 0, tzinfo=<UTC>))


In [8]:
try:
    Model(start_dt="2020/1/1", end_dt="2012/12/31")
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
end_dt
  Value error, end_dt must come after start_dt [type=value_error, input_value='2012/12/31', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


> 💡 **Complemento:** para regras que envolvem **vários** campos, o Pydantic também tem o `@model_validator(mode="after")`, que roda depois
> de todos os campos, recebe a instância pronta e não depende da ordem de declaração. A diferença prática: o erro do `field_validator`
> aponta o campo (`end_dt`), enquanto o do `model_validator` fica sem `loc` (é do modelo).

In [9]:
from pydantic import model_validator

class ModelV2(BaseModel):
    start_dt: DateTimeUTC
    end_dt: DateTimeUTC

    @model_validator(mode="after")
    def check_order(self):
        if self.end_dt <= self.start_dt:
            raise ValueError("end_dt must come after start_dt")
        return self

try:
    ModelV2(start_dt="2020/1/1", end_dt="2012/12/31")
except ValidationError as ex:
    print(f'ex.errors()[0]["loc"]: {ex.errors()[0]["loc"]}')
    print(f'ex.errors()[0]["msg"]: {ex.errors()[0]["msg"]}')

ex.errors()[0]["loc"]: ()
ex.errors()[0]["msg"]: Value error, end_dt must come after start_dt
